# Python Core Extended — I/O, Pickle, OOP, Threading

**Celebal Readiness Training**

This notebook sits between `Day3_Core_Python.ipynb` and
`Advanced_Python_Senior_SDE3.ipynb`.

| Notebook | Covers |
|---|---|
| **Day 3 Core** | syntax, strings, lists, dicts, comprehensions, basic files |
| **this one** | **file I/O in depth, pickle, OOP from scratch, threading** |
| **Advanced** | data model, descriptors, MRO internals, asyncio, free-threading |

Day 3 taught `with open(...)` and nothing about classes. The advanced notebook
assumes you already know both. **This fills that gap.**

## Contents

| § | Topic | Why |
|---|---|---|
| 1 | **File I/O in depth** | text vs binary, encodings, atomic writes, large files |
| 2 | **Pickle** | serialising Python objects — and why it's dangerous |
| 3 | **OOP from scratch to depth** | the biggest section. Classes, inheritance, SOLID |
| 4 | **Multithreading** | the GIL, locks, queues, and when threads actually help |
| 5 | Interview questions | |

## How to use it

Run top to bottom. Before submitting, **Runtime → Restart and run all**.
Cells marked **Predict first** — commit to an answer before running.


In [ ]:
import sys, os, io, json, csv, time, tempfile, shutil
from pathlib import Path

print("Python", sys.version.split()[0])

# A scratch directory for this notebook's files
WORK = Path("/content/core_extended") if Path("/content").exists() else Path("core_extended")
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print("working directory:", Path.cwd())

---

# 1. File I/O in depth

Day 3 covered `with open()`, the modes, and `csv.DictReader`. This section covers
what breaks in production.

## 1.1 Text mode vs binary mode

**This is the distinction everything else rests on**, and it's the bridge to
pickle.

In [4]:
# TEXT mode: you get str. Python DECODES bytes using an encoding.
with open("hello.txt", "w") as f:          # "w" = text mode (the default)
    f.write("Hello, world")

with open("hello.txt", "r") as f:
    text = f.read()
print("text mode :", type(text), repr(text))

# BINARY mode: you get bytes. No decoding happens at all.
with open("hello.txt", "rb") as f:         # "rb" = read binary
    raw = f.read()
print("binary    :", type(raw), repr(raw))

print()
print("Note the b'' prefix. Those are the same bytes — text mode just")
print("decoded them into characters for you.")

text mode : <class 'str'> 'Hello, world'
binary    : <class 'bytes'> b'Hello, world'

Note the b'' prefix. Those are the same bytes — text mode just
decoded them into characters for you.


| | Text mode `"r"` / `"w"` | Binary mode `"rb"` / `"wb"` |
|---|---|---|
| You get | **`str`** | **`bytes`** |
| Encoding applied | yes — decode/encode | **none** |
| Newlines translated | yes (`\r\n` → `\n`) | **no** |
| Use for | `.txt`, `.csv`, `.json`, source code | images, `.parquet`, **pickle**, anything compressed |

> **Use binary mode for any format you didn't write by hand.** Text mode will
> corrupt a PNG, a zip file or a pickle by trying to decode bytes that aren't
> characters, and by rewriting byte `0x0D` on Windows.

In [5]:
# What happens when you get it WRONG — text mode on binary data
blob = bytes(range(256))                   # every possible byte value
with open("blob.bin", "wb") as f:
    f.write(blob)

try:
    with open("blob.bin", "r") as f:       # <<< text mode. WRONG.
        f.read()
except UnicodeDecodeError as e:
    print("UnicodeDecodeError:", str(e)[:90])
    print()
    print("Byte 0x81 is not valid UTF-8, so the decode fails. In binary mode")
    print("there IS no decode step, so there is nothing to fail.")

with open("blob.bin", "rb") as f:
    print("\nbinary mode:", len(f.read()), "bytes read, no error")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 128: invalid start byte

Byte 0x81 is not valid UTF-8, so the decode fails. In binary mode
there IS no decode step, so there is nothing to fail.

binary mode: 256 bytes read, no error


## 1.2 Encodings — and the one rule

An **encoding** maps characters to bytes. `"é"` is one character but **two bytes**
in UTF-8 and **one byte** in Latin-1.

In [2]:
text = "café ☕ 日本語"

for enc in ("utf-8", "utf-16", "latin-1"):
    try:
        encoded = text.encode(enc)
        print(f"{enc:<10} {len(encoded):>3} bytes   {encoded[:28]}")
    except UnicodeEncodeError as e:
        print(f"{enc:<10} FAILS — {str(e)[:55]}")

print()
print("latin-1 cannot represent ☕ or 日本語 at all — only 256 characters exist in it.")
print("UTF-8 can represent every character, and is the correct default.")

utf-8       19 bytes   b'caf\xc3\xa9 \xe2\x98\x95 \xe6\x97\xa5\xe6\x9c\xac\xe8\xaa\x9e'
utf-16      22 bytes   b'\xff\xfec\x00a\x00f\x00\xe9\x00 \x00\x15& \x00\xe5e,g\x9e\x8a'
latin-1    FAILS — 'latin-1' codec can't encode character '\u2615' in posi

latin-1 cannot represent ☕ or 日本語 at all — only 256 characters exist in it.
UTF-8 can represent every character, and is the correct default.


In [3]:
# THE RULE: always pass encoding= explicitly.
#
# Without it, Python uses the platform default (locale.getpreferredencoding()),
# which is UTF-8 on Linux/macOS and was cp1252 on Windows until recently.
# So the SAME code produces different results on different machines.
print("this platform's default:", __import__("locale").getpreferredencoding(False))

with open("unicode.txt", "w", encoding="utf-8") as f:      # <<< explicit
    f.write("café ☕")

with open("unicode.txt", "r", encoding="utf-8") as f:
    print("read back correctly:", f.read())

# Read it with the WRONG encoding and you get mojibake, not an error
with open("unicode.txt", "r", encoding="latin-1") as f:
    print("read as latin-1    :", f.read(), "  <<< silently WRONG")

# Because Python is not trying to encode the text as Latin-1 when you read the file.
# It is decoding existing bytes as Latin-1, and every possible byte is valid in Latin-1.
# That's the key difference: encoding can fail, but decoding with the wrong encoding doesn't always fail.

this platform's default: utf-8
read back correctly: café ☕
read as latin-1    : cafÃ© â   <<< silently WRONG


> **Mojibake is worse than a crash.** `café` becoming `cafÃ©` doesn't raise —
> it just quietly corrupts your data. This is the same failure class as the
> MM/DD vs DD/MM date bug: silently wrong, not loudly broken.

### The `errors=` parameter

In [7]:
bad_bytes = b"caf\xe9 is latin-1"          # 0xe9 is not valid UTF-8

for mode in ("strict", "ignore", "replace", "backslashreplace"):
    try:
        out = bad_bytes.decode("utf-8", errors=mode)
        print(f"errors={mode:<18} {out!r}")
    except UnicodeDecodeError:
        print(f"errors={mode:<18} RAISES (this is the default)")

print()
print("USE WITH CARE:")
print("  strict   (default) — fail loudly. Correct for data you control.")
print("  replace  — insert U+FFFD. Good for logs you only need to READ.")
print("  ignore   — DROP the byte. SILENT DATA LOSS. Almost never right.")

errors=strict             RAISES (this is the default)
errors=ignore             'caf is latin-1'
errors=replace            'caf� is latin-1'
errors=backslashreplace   'caf\\xe9 is latin-1'

USE WITH CARE:
  strict   (default) — fail loudly. Correct for data you control.
  replace  — insert U+FFFD. Good for logs you only need to READ.
  ignore   — DROP the byte. SILENT DATA LOSS. Almost never right.


In [8]:
# Real-world: a file with mixed or unknown encoding.
# Try UTF-8 first, fall back, and RECORD which one worked.
def read_text_robust(path: str, encodings=("utf-8", "utf-8-sig", "cp1252", "latin-1")):
    for enc in encodings:
        try:
            with open(path, "r", encoding=enc) as f:
                return f.read(), enc
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("none of the candidate encodings worked")

# utf-8-sig strips the BOM that Excel writes at the start of CSVs
with open("bom.csv", "wb") as f:
    f.write(b"\xef\xbb\xbfname,value\nAditi,42\n")       # BOM + content

with open("bom.csv", encoding="utf-8") as f:
    print("as utf-8    :", repr(f.readline()))      # BOM leaks into the header!

with open("bom.csv", encoding="utf-8-sig") as f:
    print("as utf-8-sig:", repr(f.readline()))      # BOM stripped

print()
print("THE EXCEL BOM BUG: Excel writes a 3-byte Byte Order Mark. Read with")
print("plain utf-8 and your FIRST COLUMN NAME becomes '\ufeffname', so every")
print("df['name'] lookup fails with a KeyError. Use encoding='utf-8-sig'.")

as utf-8    : '\ufeffname,value\n'
as utf-8-sig: 'name,value\n'

THE EXCEL BOM BUG: Excel writes a 3-byte Byte Order Mark. Read with
plain utf-8 and your FIRST COLUMN NAME becomes '﻿name', so every
df['name'] lookup fails with a KeyError. Use encoding='utf-8-sig'.


## 1.3 `pathlib` — the modern way

`os.path` is string manipulation. `pathlib` is objects.

In [6]:
from pathlib import Path

p = Path("data") / "2026" / "orders.csv"      # / operator joins paths
print("path      :", p)
print("name      :", p.name)
print("stem      :", p.stem)
print("suffix    :", p.suffix)
print("parent    :", p.parent)
print("parts     :", p.parts)
print("absolute  :", p.resolve())

print()
print("comparison with os.path:")
print("  os.path.join('data','2026','orders.csv')  ->  Path('data')/'2026'/'orders.csv'")
print("  os.path.exists(p)                        ->  p.exists()")
print("  os.path.basename(p)                      ->  p.name")
print("  os.path.splitext(p)[1]                   ->  p.suffix")
print("  os.makedirs(p, exist_ok=True)            ->  p.mkdir(parents=True, exist_ok=True)")
print("  open(p).read()                           ->  p.read_text()")
print()
print("pathlib handles the separator for you, so the same code works on Windows.")

path      : data/2026/orders.csv
name      : orders.csv
stem      : orders
suffix    : .csv
parent    : data/2026
parts     : ('data', '2026', 'orders.csv')
absolute  : /content/data/2026/orders.csv

comparison with os.path:
  os.path.join('data','2026','orders.csv')  ->  Path('data')/'2026'/'orders.csv'
  os.path.exists(p)                        ->  p.exists()
  os.path.basename(p)                      ->  p.name
  os.path.splitext(p)[1]                   ->  p.suffix
  os.makedirs(p, exist_ok=True)            ->  p.mkdir(parents=True, exist_ok=True)
  open(p).read()                           ->  p.read_text()

pathlib handles the separator for you, so the same code works on Windows.


In [ ]:
# The convenience methods that remove most boilerplate
Path("quick.txt").write_text("written in one line\n", encoding="utf-8")
print(Path("quick.txt").read_text(encoding="utf-8"), end="")

Path("quick.bin").write_bytes(b"\x00\x01\x02")
print("bytes:", Path("quick.bin").read_bytes())

# Globbing
Path("sub").mkdir(exist_ok=True)
for i in range(3):
    (Path("sub") / f"file_{i}.txt").write_text(str(i))

print()
print("glob('sub/*.txt')   :", sorted(str(p) for p in Path(".").glob("sub/*.txt")))
print("rglob('*.txt')      :", sorted(str(p) for p in Path(".").rglob("*.txt"))[:5])
print()
print("rglob is RECURSIVE — it searches subdirectories. glob does not.")

## 1.4 Reading files larger than memory

The single most important file-handling skill.

In [9]:
# Build a file with 200,000 lines
with open("big.csv", "w", encoding="utf-8") as f:
    f.write("id,region,amount\n")
    for i in range(200_000):
        f.write(f"{i},{'WEST' if i % 2 else 'EAST'},{i * 1.5:.2f}\n")

size_mb = Path("big.csv").stat().st_size / 1e6
print(f"file size: {size_mb:.1f} MB, 200,000 rows")

file size: 4.2 MB, 200,000 rows


In [10]:
import tracemalloc

# ---- WRONG: read everything into memory ----
tracemalloc.start()
with open("big.csv", encoding="utf-8") as f:
    all_lines = f.readlines()                      # <<< the whole file as a list
peak_bad = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()
del all_lines

# ---- RIGHT: iterate the file object ----
tracemalloc.start()
total = 0.0
with open("big.csv", encoding="utf-8") as f:
    next(f)                                        # skip the header
    for line in f:                                 # <<< ONE LINE AT A TIME
        total += float(line.rsplit(",", 1)[1])
peak_good = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

print(f"readlines()   peak memory: {peak_bad/1e6:>8.2f} MB")
print(f"iterate file  peak memory: {peak_good/1e6:>8.2f} MB")
print(f"ratio                    : {peak_bad/peak_good:>8.0f}x")
print(f"\ntotal computed: {total:,.2f}")
print()
print("`for line in f` is LAZY. Memory stays constant no matter how big the")
print("file is — 2 MB or 2 TB. readlines() needs the whole thing at once.")

readlines()   peak memory:    14.06 MB
iterate file  peak memory:     0.03 MB
ratio                    :      476x

total computed: 29,999,850,000.00

`for line in f` is LAZY. Memory stays constant no matter how big the
file is — 2 MB or 2 TB. readlines() needs the whole thing at once.


In [ ]:
# Reading in fixed-size CHUNKS — for binary files or when lines are irrelevant
def read_chunks(path, chunk_size=64 * 1024):
    """Yield the file in chunks. The two-argument form of iter() is the idiom."""
    with open(path, "rb") as f:
        # iter(callable, sentinel) calls f.read(n) until it returns b""
        yield from iter(lambda: f.read(chunk_size), b"")

import hashlib
h = hashlib.md5()
n_chunks = 0
for chunk in read_chunks("big.csv"):
    h.update(chunk)
    n_chunks += 1

print(f"hashed {n_chunks} chunks of 64 KB")
print(f"md5: {h.hexdigest()}")
print()
print("This is how you checksum a 500 GB file on a 4 GB machine.")

## 1.5 Atomic writes — the pattern nobody teaches

**The problem:** your process writes `output.csv` and crashes halfway. Now the
file exists, looks valid, and is **truncated**. A downstream reader can't tell.

In [ ]:
# ---- THE BUG ----
def unsafe_write(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for i, r in enumerate(rows):
            if i == 3:
                raise RuntimeError("crashed mid-write")
            f.write(r + "\n")

try:
    unsafe_write("unsafe.txt", [f"row {i}" for i in range(10)])
except RuntimeError as e:
    print("crash:", e)

print("file exists:", Path("unsafe.txt").exists())
print("contents   :", repr(Path("unsafe.txt").read_text()))
print()
print("A PARTIAL FILE that looks complete. Nothing errors downstream.")

In [ ]:
# ---- THE FIX: write to a temp file, then RENAME ----
#
# On the same filesystem, os.replace() is ATOMIC: the destination either has
# the old content or the new content. There is no visible in-between state.
def atomic_write(path, rows):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    try:
        with open(tmp, "w", encoding="utf-8") as f:
            for i, r in enumerate(rows):
                if i == 3:
                    raise RuntimeError("crashed mid-write")
                f.write(r + "\n")
            f.flush()
            os.fsync(f.fileno())        # force the OS to write to DISK, not cache
        os.replace(tmp, path)           # <<< ATOMIC
    except Exception:
        tmp.unlink(missing_ok=True)     # clean up the partial temp file
        raise

Path("safe.txt").write_text("the PREVIOUS good version\n")

try:
    atomic_write("safe.txt", [f"row {i}" for i in range(10)])
except RuntimeError as e:
    print("crash:", e)

print("contents after the crash:", repr(Path("safe.txt").read_text()))
print("temp file left behind   :", Path("safe.txt.tmp").exists())
print()
print("The OLD version survived INTACT. A reader never sees a partial file.")
print("This is the same idea as Day 5's 'overwrite the partition, never append'.")

> **`os.replace` vs `os.rename`:** `replace` overwrites an existing destination
> and is atomic on both POSIX and Windows. `rename` fails if the destination
> exists on Windows. Use `os.replace`.
>
> **`fsync` matters** if the machine could lose power. Without it, the data may
> sit in the OS cache while the rename has already happened.

## 1.6 `seek` and `tell` — random access

In [ ]:
with open("seekable.txt", "w", encoding="utf-8") as f:
    f.write("0123456789ABCDEFGHIJ")

with open("seekable.txt", "r", encoding="utf-8") as f:
    print("position at open   :", f.tell())
    print("read 5             :", f.read(5), " -> position", f.tell())
    f.seek(10)                                     # jump to byte 10
    print("after seek(10)     :", f.read(5))
    f.seek(0)                                      # rewind
    print("after seek(0)      :", f.read(3))

# Whence: 0=start (default), 1=current, 2=end. Non-zero whence needs BINARY mode.
with open("seekable.txt", "rb") as f:
    f.seek(-5, 2)                                  # 5 bytes before the END
    print("last 5 bytes       :", f.read())

print()
print("USE FOR: reading a file header, tailing a log, fixed-width records,")
print("         resuming an interrupted download.")
print("NOT FOR: normal sequential reading — just iterate.")

## 1.7 `ExitStack` — a variable number of files

You cannot write `with open(a), open(b), open(c)` when the count is dynamic.

In [ ]:
from contextlib import ExitStack

paths = [f"part_{i}.txt" for i in range(4)]
for i, p in enumerate(paths):
    Path(p).write_text(f"content of part {i}\n")

# Open ALL of them, guaranteed to close in reverse order even if one raises
with ExitStack() as stack:
    files = [stack.enter_context(open(p, encoding="utf-8")) for p in paths]
    print("opened", len(files), "files")
    for f in files:
        print("  ", f.readline().strip())

print("all closed:", all(f.closed for f in files))

In [ ]:
# A practical use: merge many files into one, streaming
with ExitStack() as stack, open("merged.txt", "w", encoding="utf-8") as out:
    for p in paths:
        f = stack.enter_context(open(p, encoding="utf-8"))
        shutil.copyfileobj(f, out)         # streams in chunks, no full read

print(Path("merged.txt").read_text(), end="")
print()
print("shutil.copyfileobj streams in 64 KB chunks — it never loads a whole file.")

## 1.8 Temporary files

In [ ]:
import tempfile

# A named temp file that persists until you delete it
with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=False,
                                 encoding="utf-8") as tf:
    tf.write("id,value\n1,100\n")
    tmp_name = tf.name
print("temp file:", tmp_name)
print("contents :", Path(tmp_name).read_text().strip())
os.unlink(tmp_name)

# A whole temp DIRECTORY, cleaned up automatically
with tempfile.TemporaryDirectory() as tmpdir:
    p = Path(tmpdir) / "scratch.txt"
    p.write_text("gone when the block exits")
    print("inside the block, exists:", p.exists())
print("after the block, dir exists:", Path(tmpdir).exists())

print()
print("USE TemporaryDirectory for: intermediate pipeline output, test fixtures,")
print("extracting an archive you only need once.")

## 1.9 Section 1 recap

| Topic | The rule |
|---|---|
| Text vs binary | binary for anything you didn't hand-write |
| Encoding | **always pass `encoding="utf-8"` explicitly** |
| Excel CSVs | use `encoding="utf-8-sig"` to strip the BOM |
| `errors="ignore"` | silent data loss — avoid |
| Large files | `for line in f`, never `readlines()` |
| Binary streaming | `iter(lambda: f.read(n), b"")` |
| Crash safety | write to `.tmp`, `fsync`, then `os.replace` |
| Many files | `ExitStack` |
| Paths | `pathlib`, not `os.path` |
| Scratch space | `tempfile.TemporaryDirectory` |

**The two that cause real incidents:** a missing `encoding=` (same code, different
result per machine) and a non-atomic write (partial files that look complete).

---

# 2. Pickle

## 2.1 The problem it solves

You have a Python object in memory. You want it back **after the process ends**.

JSON handles dicts, lists, strings and numbers. It cannot handle a trained
sklearn model, a `datetime`, a `set`, a custom class, or a nested object graph.

**Pickle serialises almost any Python object into bytes, and reconstructs it.**

In [ ]:
import pickle
from datetime import datetime, date
from collections import Counter, defaultdict, deque

obj = {
    "when": datetime(2026, 9, 1, 14, 30),
    "tags": {"a", "b", "c"},                    # a SET
    "counts": Counter("hello world"),
    "queue": deque([1, 2, 3]),
    "nested": {"deep": [1, {"deeper": (2, 3)}]},   # a TUPLE inside
    "none": None,
}

# dumps -> bytes.  loads -> object.  (the 's' means STRING/bytes)
blob = pickle.dumps(obj)
back = pickle.loads(blob)

print(f"pickled to {len(blob)} bytes")
print("round trip identical:", back == obj)
print()
for k, v in back.items():
    print(f"  {k:<8} {type(v).__name__:<12} {v}")

In [ ]:
# The same thing, JSON — and why it fails
try:
    json.dumps(obj)
except TypeError as e:
    print("json.dumps:", e)
print()
print("JSON has no concept of a set, a datetime, a Counter or a deque.")
print("It knows exactly: object, array, string, number, true, false, null.")

## 2.2 `dump` / `load` vs `dumps` / `loads`

| Function | Direction | Target |
|---|---|---|
| `pickle.dump(obj, file)` | write | **a file** |
| `pickle.load(file)` | read | a file |
| `pickle.dumps(obj)` | write | **bytes** (the `s` = string) |
| `pickle.loads(data)` | read | bytes |

Same naming convention as `json`. The `s` suffix means "to/from a string of
bytes rather than a file".

> **Pickle files MUST be opened in binary mode.** `"wb"` and `"rb"`. Text mode
> will corrupt the bytes.

In [ ]:
# To a file
with open("data.pkl", "wb") as f:              # <<< "wb", not "w"
    pickle.dump(obj, f)

with open("data.pkl", "rb") as f:              # <<< "rb", not "r"
    loaded = pickle.load(f)

print("from file, identical:", loaded == obj)
print("file size:", Path("data.pkl").stat().st_size, "bytes")

# What happens in TEXT mode
try:
    with open("data_text.pkl", "w") as f:
        pickle.dump(obj, f)
except TypeError as e:
    print("\ntext mode:", e)
    print("  pickle produces BYTES; a text-mode file expects str.")

## 2.3 Protocols

Pickle has six format versions. Higher = faster and smaller, but less portable.

In [ ]:
print("default protocol  :", pickle.DEFAULT_PROTOCOL)
print("highest available :", pickle.HIGHEST_PROTOCOL)
print()

big = {"numbers": list(range(50_000)), "text": ["row"] * 10_000}

for proto in range(pickle.HIGHEST_PROTOCOL + 1):
    t0 = time.perf_counter()
    data = pickle.dumps(big, protocol=proto)
    elapsed = time.perf_counter() - t0
    print(f"  protocol {proto}: {len(data):>9,} bytes   {elapsed*1000:>6.1f} ms")

| Protocol | Added | Notes |
|---|---|---|
| 0 | ancient | ASCII, human-readable, very slow |
| 1, 2 | Python 2 | binary; 2 added new-style classes |
| **3** | Python 3.0 | first Python-3-only; handles `bytes` |
| **4** | Python 3.4 | large objects, **the default in 3.4–3.7** |
| **5** | Python 3.8 | out-of-band buffers (zero-copy for NumPy) |

**Practical advice:** leave the protocol alone. The default tracks your Python
version. Only set it explicitly when an **older** Python must read the file:

```python
pickle.dumps(obj, protocol=4)      # readable by Python 3.4+
```

## 2.4 What cannot be pickled

In [ ]:
def module_level_function(x):
    return x * 2


class SimpleClass:
    def __init__(self, v): self.v = v


cases = [
    ("int",                 42),
    ("list",                [1, 2, 3]),
    ("dict",                {"a": 1}),
    ("set",                 {1, 2}),
    ("datetime",            datetime.now()),
    ("custom instance",     SimpleClass(5)),
    ("module-level func",   module_level_function),
    ("built-in func",       len),
    ("lambda",              lambda x: x),
    ("closure",             (lambda: (lambda y: y))()),
    ("generator",           (i for i in range(3))),
    ("open file handle",    open("hello.txt")),
    ("thread lock",         __import__("threading").Lock()),
]

for name, value in cases:
    try:
        pickle.dumps(value)
        print(f"  OK      {name}")
    except Exception as e:
        print(f"  FAILS   {name:<22} {type(e).__name__}")

cases[-3][1].close()

### Why functions "work" but lambdas don't

Pickle does **not** store a function's code. It stores its **import path**:

```
module_level_function  →  pickled as "__main__.module_level_function"
```

On unpickling, Python *imports that name*. So:

- **Module-level functions and classes work** — they have a findable name
- **Lambdas fail** — they're all called `<lambda>`, so there's nothing to look up
- **Closures fail** — the enclosed variables have no importable name
- **File handles, sockets, locks fail** — they're OS resources, not data

> ### The consequence that bites people
>
> **The class definition must be importable when you unpickle.** Pickle a
> `SimpleClass` instance, then rename the class or move the module, and
> `pickle.load` raises `AttributeError` or `ModuleNotFoundError`.
>
> **A pickle is not self-contained.** It's a reference to code plus the data.

## 2.5 Controlling it — `__getstate__` and `__setstate__`

When your object holds something unpicklable, define what gets saved.

In [ ]:
import threading


class DatabaseClient:
    """Holds a lock and a 'connection' — neither can be pickled."""

    def __init__(self, dsn):
        self.dsn = dsn
        self.query_count = 0
        self._lock = threading.Lock()          # NOT picklable
        self._connection = open("hello.txt")   # NOT picklable

    def __repr__(self):
        return (f"DatabaseClient({self.dsn!r}, queries={self.query_count}, "
                f"connected={self._connection is not None})")


client = DatabaseClient("mysql://localhost/classicmodels")
client.query_count = 17

try:
    pickle.dumps(client)
    print("unexpectedly succeeded")
except Exception as e:
    print(f"without __getstate__ -> {type(e).__name__}: {str(e)[:80]}")
    print("  (a threading.Lock holds OS state; there is nothing to serialise)")
client._connection.close()

In [ ]:
class PicklableClient:
    """The same object, made picklable by declaring what to save."""

    def __init__(self, dsn):
        self.dsn = dsn
        self.query_count = 0
        self._connect()

    def _connect(self):
        self._lock = threading.Lock()
        self._connection = f"<live connection to {self.dsn}>"

    # ---- called ON PICKLING: return what should be saved -----------------
    def __getstate__(self):
        state = self.__dict__.copy()           # start from everything
        del state["_lock"]                     # drop the unpicklable bits
        del state["_connection"]
        return state

    # ---- called ON UNPICKLING: rebuild from the saved state --------------
    def __setstate__(self, state):
        self.__dict__.update(state)            # restore the saved attributes
        self._connect()                        # RE-CREATE the transient ones

    def __repr__(self):
        return (f"PicklableClient({self.dsn!r}, queries={self.query_count}, "
                f"conn={self._connection})")


c = PicklableClient("mysql://localhost/classicmodels")
c.query_count = 17
print("original :", c)

try:
    restored = pickle.loads(pickle.dumps(c))
    print("restored :", restored)
except Exception as e:
    # Only happens if the class isn't importable from __main__ — e.g. inside
    # certain sandboxed runners. In a normal notebook or script it works.
    print(f"skipped  : {type(e).__name__} — the class must be IMPORTABLE for")
    print("           pickle to find it on load. See 2.4 on why.")
print()
print("The DSN and query_count survived. The lock and connection were")
print("REBUILT on load rather than serialised — which is the only sane option,")
print("since a connection to a database that may no longer exist is meaningless.")

> **The pattern:** `__getstate__` returns a dict of what to save.
> `__setstate__` receives it and reconstructs anything transient.
>
> This is exactly how sklearn models, PyTorch modules and database clients
> handle serialisation internally.

## 2.6 Security — the part that actually matters

> ### NEVER UNPICKLE DATA YOU DID NOT CREATE
>
> Unpickling **executes code**. It is not a data format; it is a small program
> that the unpickler runs.

In [ ]:
# A demonstration. This is deliberately harmless, but the mechanism is real.
class Exploit:
    def __reduce__(self):
        # __reduce__ tells pickle HOW to reconstruct the object:
        #   return (callable, args)  ->  on load, pickle calls callable(*args)
        import os
        return (os.system, ('echo "ARBITRARY CODE EXECUTED on unpickle"',))


malicious = pickle.dumps(Exploit())
print(f"the payload is {len(malicious)} bytes of innocent-looking data\n")

print("now loading it:")
pickle.loads(malicious)          # <<< runs os.system(...)

print()
print("No warning. No sandbox. No confirmation. It simply ran.")
print("Replace echo with anything — curl a script, read ~/.ssh, exfiltrate data.")

### Where this goes wrong in practice

| Scenario | Risk |
|---|---|
| A `.pkl` model downloaded from the internet | **arbitrary code execution** |
| Pickle in a Redis cache an attacker can write to | ACE |
| Pickle as a session cookie | ACE |
| Pickle over a network socket | ACE |
| A model file from a colleague's laptop | depends entirely on trust |

### What to use instead

| Need | Use |
|---|---|
| Config, API payloads | **JSON** |
| Tabular data | **Parquet** / CSV |
| Cross-language | **Protocol Buffers**, MessagePack, Avro |
| ML models, long-term | **ONNX**, or the framework's own format |
| Numeric arrays | `numpy.save` / `.npz` |
| Untrusted input | **anything but pickle** |

> **The one-line rule:** pickle is fine for *your own process talking to itself*
> — caching, `multiprocessing`, checkpointing. It is **never** an interchange
> format and **never** safe for input you didn't produce.

## 2.7 Pickle vs the alternatives, measured

In [ ]:
import marshal

records = [{"id": i, "name": f"customer-{i}", "amount": i * 1.5,
            "active": i % 2 == 0} for i in range(20_000)]

results = []

# pickle
t0 = time.perf_counter(); p = pickle.dumps(records); t_dump = time.perf_counter() - t0
t0 = time.perf_counter(); pickle.loads(p);            t_load = time.perf_counter() - t0
results.append(("pickle", len(p), t_dump, t_load))

# json
t0 = time.perf_counter(); j = json.dumps(records).encode(); t_dump = time.perf_counter() - t0
t0 = time.perf_counter(); json.loads(j);                    t_load = time.perf_counter() - t0
results.append(("json", len(j), t_dump, t_load))

# marshal — CPython's internal format. FAST, but version-specific and unsafe.
t0 = time.perf_counter(); m = marshal.dumps(records); t_dump = time.perf_counter() - t0
t0 = time.perf_counter(); marshal.loads(m);           t_load = time.perf_counter() - t0
results.append(("marshal", len(m), t_dump, t_load))

print(f"{'format':<10}{'bytes':>12}{'dump ms':>10}{'load ms':>10}")
print("-" * 42)
for name, size, td, tl in results:
    print(f"{name:<10}{size:>12,}{td*1000:>10.1f}{tl*1000:>10.1f}")

print()
print("NOTES")
print("  pickle   handles any Python object. UNSAFE on untrusted input.")
print("  json     text, human-readable, cross-language. Limited types.")
print("  marshal  fastest, but the format changes between Python versions")
print("           and is explicitly documented as not for untrusted data.")
print("           It exists for .pyc files. Do not use it in your code.")

## 2.8 `joblib` — the practical choice for ML

sklearn's own documentation recommends `joblib` over bare `pickle` for models,
because it stores large NumPy arrays far more efficiently.

```python
import joblib

joblib.dump(model, "churn_model.joblib")       # instead of pickle.dump
model = joblib.load("churn_model.joblib")

joblib.dump(model, "churn_model.joblib.gz", compress=3)   # with compression
```

**The security caveat is identical** — `joblib.load` also executes code. It's a
pickle wrapper, not a safer format.

> **For production model serving**, the recommendation is to move off pickle
> entirely: ONNX for cross-framework portability, or re-fit from versioned code
> plus versioned data. Day 5's model card plus Delta time travel is that pattern.

## 2.9 Section 2 recap

| Point | Detail |
|---|---|
| What it does | serialises almost any Python object to bytes |
| File mode | **`"wb"` / `"rb"` — always binary** |
| Protocol | leave at default unless an older Python must read it |
| Can't pickle | lambdas, closures, generators, file handles, locks, sockets |
| Why | pickle stores an **import path**, not code |
| Custom control | `__getstate__` / `__setstate__` |
| **Security** | **unpickling EXECUTES CODE. Never on untrusted data.** |
| Use it for | caching, `multiprocessing`, checkpointing — your own process |
| Don't use it for | interchange, APIs, sessions, anything from outside |
| For ML | `joblib` (same risk), or ONNX for production |

---

# 3. OOP from scratch to depth

The largest section. Day 3 taught functions; this teaches classes.

## 3.1 The problem OOP solves

Start without classes. Model a bank account using a dict and some functions.

In [11]:
# ---- THE PROCEDURAL VERSION ----
def make_account(owner, balance=0.0):
    return {"owner": owner, "balance": balance, "history": []}

def deposit(account, amount):
    account["balance"] += amount
    account["history"].append(("deposit", amount))

def withdraw(account, amount):
    account["balance"] -= amount
    account["history"].append(("withdraw", amount))

acct = make_account("Aditi", 1000)
deposit(acct, 500)
withdraw(acct, 200)
print(acct["balance"])

# It works. Now watch what nothing stops you doing:
acct["balance"] = 999_999_999            # <<< no deposit, no history. Just set it.
acct["blance"] = 50                      # <<< typo creates a NEW key, silently
del acct["history"]                      # <<< remove a field the functions need

print("balance after tampering:", acct["balance"])
print("keys now:", list(acct.keys()))
print()
print("THREE PROBLEMS:")
print("  1. the data and the functions that manage it are SEPARATE")
print("  2. nothing enforces the rules — anyone can set balance directly")
print("  3. a typo creates a new field instead of raising")

1300
balance after tampering: 999999999
keys now: ['owner', 'balance', 'blance']

THREE PROBLEMS:
  1. the data and the functions that manage it are SEPARATE
  2. nothing enforces the rules — anyone can set balance directly
  3. a typo creates a new field instead of raising


**A class bundles the data and the operations together, and controls access
to the data.** That's the whole idea. Everything else is detail.

## 3.2 Your first class

In [ ]:
class Account:
    """A bank account. Data and behaviour in one place."""

    def __init__(self, owner, balance=0.0):
        # __init__ runs when you create an instance. It sets up the object.
        # `self` is THIS instance — the one being created.
        self.owner = owner
        self.balance = balance
        self.history = []

    def deposit(self, amount):
        self.balance += amount
        self.history.append(("deposit", amount))

    def withdraw(self, amount):
        self.balance -= amount
        self.history.append(("withdraw", amount))


acct = Account("Aditi", 1000)       # calling the class CREATES an instance
acct.deposit(500)
acct.withdraw(200)

print("owner   :", acct.owner)
print("balance :", acct.balance)
print("history :", acct.history)
print()
print("type(acct)   :", type(acct))
print("is instance  :", isinstance(acct, Account))

### The vocabulary

| Term | Means | Here |
|---|---|---|
| **class** | the blueprint | `Account` |
| **instance** / object | one thing built from it | `acct` |
| **attribute** | data on the instance | `acct.balance` |
| **method** | a function on the class | `acct.deposit()` |
| **`__init__`** | the initialiser, runs on creation | sets owner/balance |
| **`self`** | the instance the method was called on | — |

### What `self` actually is

`self` is **not magic**. It's just the first parameter, and Python passes the
instance into it automatically.

In [ ]:
# These two lines are IDENTICAL:
acct.deposit(100)              # the usual way
Account.deposit(acct, 100)     # what Python actually does

print("balance:", acct.balance)
print()
print("acct.deposit(100)  BECOMES  Account.deposit(acct, 100)")
print()
print("`self` is just the name convention for that first argument. You COULD")
print("call it anything — but don't. Every Python programmer expects `self`.")

## 3.3 Instance vs class attributes — and the trap

**The single most common OOP bug in Python.**

### Predict first

Each `Account` gets its own `history` list, right? What does this print?

In [ ]:
class BrokenAccount:
    history = []                       # <<< defined in the CLASS BODY

    def __init__(self, owner):
        self.owner = owner

    def deposit(self, amount):
        self.history.append((self.owner, amount))


a = BrokenAccount("Aditi")
b = BrokenAccount("Rahul")

a.deposit(100)
b.deposit(500)

print("a.history:", a.history)
print("b.history:", b.history)
print("same object:", a.history is b.history)

**One list, shared by every instance.** `a.deposit()` appended to the same
list `b` uses.

**Why:** the class body executes **once**, when the class is defined. So
`history = []` creates exactly one list, stored on the class. Every instance
sees it.

In [ ]:
class FixedAccount:
    bank_name = "Celebal Bank"         # CLASS attribute — shared, and that's CORRECT
    interest_rate = 0.045              # same for every account

    def __init__(self, owner):
        self.owner = owner             # INSTANCE attribute — per object
        self.history = []              # <<< created fresh in __init__

    def deposit(self, amount):
        self.history.append((self.owner, amount))


a = FixedAccount("Aditi"); b = FixedAccount("Rahul")
a.deposit(100); b.deposit(500)

print("a.history:", a.history)
print("b.history:", b.history)
print("same object:", a.history is b.history)
print()
print("shared class attributes (correct use):")
print("  a.bank_name:", a.bank_name, " b.bank_name:", b.bank_name)
print("  change it on the CLASS and every instance sees it:")
FixedAccount.interest_rate = 0.05
print("  a.interest_rate:", a.interest_rate, " b.interest_rate:", b.interest_rate)

| Attribute type | Defined | Shared? | Use for |
|---|---|---|---|
| **Class attribute** | in the class body | **yes, by all instances** | constants, config, counters |
| **Instance attribute** | in `__init__`, via `self.` | no, one per object | **anything mutable** |

> **The rule: never put a mutable default (`[]`, `{}`, `set()`) in the class
> body.** It's the same bug as a mutable default argument — the object is
> created once and reused forever.

## 3.4 Encapsulation — controlling access

Back to problem 2 from §3.1: nothing stopped `acct["balance"] = 999999999`.

In [ ]:
class SafeAccount:
    def __init__(self, owner, balance=0.0):
        self.owner = owner              # public — anyone may read/write
        self._bank_code = "CEL001"      # _protected  — "internal, please don't"
        self.__balance = balance        # __private   — name-mangled
        self.history = []

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.__balance += amount
        self.history.append(("deposit", amount))

    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("withdrawal must be positive")
        if amount > self.__balance:
            raise ValueError(f"insufficient funds: have {self.__balance}, "
                             f"requested {amount}")
        self.__balance -= amount
        self.history.append(("withdraw", amount))

    def get_balance(self):
        return self.__balance


acct = SafeAccount("Aditi", 1000)
acct.deposit(500)
print("balance:", acct.get_balance())

for bad in [("deposit", -50), ("withdraw", 10_000)]:
    try:
        getattr(acct, bad[0])(bad[1])
    except ValueError as e:
        print(f"  {bad[0]}({bad[1]}) -> ValueError: {e}")

In [ ]:
# The three conventions, and what they actually do
print("public      acct.owner        ->", acct.owner)
print("protected   acct._bank_code   ->", acct._bank_code, "  (accessible! just a convention)")

try:
    print(acct.__balance)
except AttributeError as e:
    print("private     acct.__balance    -> AttributeError:", e)

print()
print("But it isn't really private — Python MANGLED the name:")
print("  actual attribute:", [a for a in vars(acct) if "balance" in a])
print("  reachable via   :", acct._SafeAccount__balance)
print()
print("NAME MANGLING: inside a class, __x becomes _ClassName__x.")
print("Its PURPOSE is to avoid accidental collisions in subclasses,")
print("NOT to provide security. Python has no truly private attributes.")

| Convention | Meaning | Enforced? |
|---|---|---|
| `name` | public API | — |
| `_name` | internal, don't rely on it | **no** — pure convention |
| `__name` | name-mangled to `_Class__name` | partially — avoids subclass collisions |
| `__name__` | dunder, Python's own protocol | don't invent your own |

> **"We're all consenting adults here"** is the Python philosophy. Use `_name`
> to signal intent. Use `__name` only when you genuinely need to avoid a
> subclass clobbering an attribute.

## 3.5 Properties — the Pythonic alternative to getters

`get_balance()` works, but Java-style getters aren't idiomatic Python.

In [ ]:
class Temperature:
    def __init__(self, celsius=0.0):
        self.celsius = celsius          # this goes THROUGH the setter below

    @property
    def celsius(self):
        """Read like an attribute, computed like a method."""
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError(f"{value} is below absolute zero")
        self._celsius = float(value)

    @property
    def fahrenheit(self):
        """COMPUTED and read-only — there's no setter."""
        return self._celsius * 9 / 5 + 32


t = Temperature(25)
print("celsius   :", t.celsius)          # looks like an attribute
print("fahrenheit:", t.fahrenheit)       # computed on access

t.celsius = 30                            # looks like assignment, runs validation
print("after set :", t.celsius, "->", t.fahrenheit)

try:
    t.celsius = -300
except ValueError as e:
    print("validation:", e)

try:
    t.fahrenheit = 100
except AttributeError as e:
    print("read-only :", e)

> ### Don't add a property until you need one
>
> Start with a plain attribute. If you later need validation or computation,
> converting it to a property is a **non-breaking change** — callers still write
> `obj.celsius`.
>
> That's exactly why Python has no getter/setter convention: you can add the
> behaviour later without changing the interface.

## 3.6 The three method types

In [ ]:
from datetime import date


class Order:
    TAX_RATE = 0.18                     # class attribute
    _count = 0                          # a counter across all instances

    def __init__(self, order_id, lines, placed=None):
        self.order_id = order_id
        self.lines = lines
        self.placed = placed or date.today()
        Order._count += 1               # note: Order._count, not self._count

    # ---- INSTANCE METHOD: operates on THIS object's data ----------------
    def total(self):
        return sum(qty * price for qty, price in self.lines)

    def total_with_tax(self):
        return round(self.total() * (1 + self.TAX_RATE), 2)

    # ---- CLASSMETHOD: receives the CLASS. Alternative constructors. -----
    @classmethod
    def from_csv_row(cls, row):
        """cls means subclasses get instances of THEMSELVES, not of Order."""
        oid, *rest = row.split(",")
        lines = [(int(rest[i]), float(rest[i + 1])) for i in range(0, len(rest), 2)]
        return cls(oid, lines)

    @classmethod
    def empty(cls, order_id):
        return cls(order_id, [])

    @classmethod
    def count(cls):
        return cls._count

    # ---- STATICMETHOD: neither self nor cls. Namespaced helper. ---------
    @staticmethod
    def is_valid_id(order_id):
        return isinstance(order_id, str) and order_id.startswith("ORD-")

    def __repr__(self):
        return f"{type(self).__name__}({self.order_id!r}, {len(self.lines)} lines)"


o = Order("ORD-001", [(2, 100.0), (1, 250.0)])
print(o, "total", o.total(), "with tax", o.total_with_tax())

print("from_csv_row :", Order.from_csv_row("ORD-002,3,50.0,1,80.0"))
print("empty        :", Order.empty("ORD-003"))
print("is_valid_id  :", Order.is_valid_id("ORD-004"), Order.is_valid_id("xyz"))
print("count        :", Order.count())

In [ ]:
# WHY classmethod uses `cls` and not the hard-coded class name
class PriorityOrder(Order):
    pass

print("Order.from_csv_row      ->", type(Order.from_csv_row("ORD-005,1,10.0")).__name__)
print("PriorityOrder.from_csv_row ->",
      type(PriorityOrder.from_csv_row("ORD-006,1,10.0")).__name__)
print()
print("The subclass got a PriorityOrder, not an Order — because the method")
print("uses cls(...) rather than Order(...). Hard-coding the name would break")
print("every subclass. THAT is why alternative constructors use classmethod.")

| | Receives | Subclass-aware | Use for |
|---|---|---|---|
| instance method | `self` | n/a | behaviour on the object's data |
| **`@classmethod`** | `cls` | **yes** | alternative constructors, registries, counters |
| **`@staticmethod`** | nothing | no | a helper that belongs in this namespace |
| plain function | nothing | no | **the default** — don't reach for staticmethod reflexively |

## 3.7 Inheritance

In [ ]:
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def describe(self):
        return f"{self.name}, earning {self.salary:,.0f}"

    def annual_cost(self):
        return self.salary * 12


class Manager(Employee):                      # Manager INHERITS from Employee
    def __init__(self, name, salary, reports):
        super().__init__(name, salary)        # <<< call the parent's __init__
        self.reports = reports                # then add our own

    # OVERRIDE: same method name, different behaviour
    def describe(self):
        base = super().describe()             # reuse the parent's version
        return f"{base}, managing {len(self.reports)} people"

    # EXTEND: a method the parent doesn't have
    def add_report(self, employee):
        self.reports.append(employee)


e = Employee("Rahul", 50_000)
m = Manager("Aditi", 90_000, [e])

print(e.describe())
print(m.describe())
print()
print("inherited method  :", m.annual_cost())      # not defined on Manager
print("isinstance Manager:", isinstance(m, Manager))
print("isinstance Employee:", isinstance(m, Employee), " <<< a Manager IS an Employee")
print("issubclass        :", issubclass(Manager, Employee))

### `super()` — and why you must call it

`super().__init__(...)` runs the parent's initialiser. **Forget it and the
parent's attributes never get set.**

In [ ]:
class BrokenManager(Employee):
    def __init__(self, name, salary, reports):
        # super().__init__(name, salary)    <<< FORGOTTEN
        self.reports = reports

bm = BrokenManager("Sneha", 80_000, [])
print("has reports:", hasattr(bm, "reports"))
try:
    bm.describe()
except AttributeError as e:
    print("describe() -> AttributeError:", e)
print()
print("`name` and `salary` were never set, so any inherited method that")
print("uses them fails. ALWAYS call super().__init__() first.")

## 3.8 Polymorphism and duck typing

**Polymorphism:** different classes, same method name, caller doesn't care which.

In [ ]:
class Dog:
    def speak(self): return "Woof"

class Cat:
    def speak(self): return "Meow"

class Robot:
    def speak(self): return "BEEP"       # NOT related to Dog or Cat at all


# One function, any object with .speak()
def make_it_talk(thing):
    print(f"  {type(thing).__name__:<8} says {thing.speak()}")

for obj in (Dog(), Cat(), Robot()):
    make_it_talk(obj)

print()
print("DUCK TYPING: 'if it walks like a duck and quacks like a duck, it's a duck.'")
print("Robot shares NO base class with Dog, yet works — because Python checks")
print("whether the METHOD EXISTS, not what the type is.")
print()
print("In Java you'd need a Speaker interface and all three would implement it.")
print("Python needs nothing. That's the difference between NOMINAL typing")
print("(by declared type) and STRUCTURAL typing (by shape).")

## 3.9 Abstract base classes — enforcing the contract

Duck typing is flexible, but sometimes you want to **guarantee** a subclass
implements something.

In [ ]:
from abc import ABC, abstractmethod


class PaymentMethod(ABC):
    """An abstract base class. Cannot be instantiated directly."""

    @abstractmethod
    def pay(self, amount):
        """Subclasses MUST implement this."""

    @abstractmethod
    def refund(self, amount):
        """And this."""

    # A CONCRETE method built on the abstract ones — the value an ABC adds
    def pay_with_receipt(self, amount):
        result = self.pay(amount)
        return f"{result}  [receipt #{abs(hash((type(self).__name__, amount))) % 10000:04d}]"


class CreditCard(PaymentMethod):
    def pay(self, amount):    return f"charged {amount:,.2f} to card"
    def refund(self, amount): return f"refunded {amount:,.2f} to card"


class UPI(PaymentMethod):
    def pay(self, amount):    return f"collected {amount:,.2f} via UPI"
    def refund(self, amount): return f"returned {amount:,.2f} via UPI"


for method in (CreditCard(), UPI()):
    print(method.pay_with_receipt(1500))

# Instantiating the ABC itself FAILS
try:
    PaymentMethod()
except TypeError as e:
    print("\nPaymentMethod() ->", e)

# And so does an INCOMPLETE subclass
class Incomplete(PaymentMethod):
    def pay(self, amount): return "paid"
    # refund is missing

try:
    Incomplete()
except TypeError as e:
    print("Incomplete()    ->", e)

> **ABCs fail at instantiation, not at definition.** `class Incomplete(...)` is
> legal; `Incomplete()` is not. That's deliberate — it lets you build partial
> hierarchies.
>
> **When to use an ABC vs duck typing:** reach for an ABC when you're defining a
> base others will *extend* and you want to share concrete code. Stick with duck
> typing (or `typing.Protocol`) when you're defining what you *consume*.

## 3.10 Multiple inheritance and the MRO

A class can inherit from several parents. Python resolves conflicts with the
**Method Resolution Order**.

In [ ]:
class Base:
    def who(self): return "Base"

class Left(Base):
    def who(self): return "Left -> " + super().who()

class Right(Base):
    def who(self): return "Right -> " + super().who()

class Both(Left, Right):
    def who(self): return "Both -> " + super().who()


print("MRO:", [c.__name__ for c in Both.__mro__])
print()
print(Both().who())
print()
print("NOTE: inside Left.who, super() resolves to RIGHT — not Base.")
print("super() means 'the NEXT class in the MRO of type(self)',")
print("NOT 'my parent'. That's why Base.who runs exactly ONCE.")
print()
print("Compare Left on its own:")
print("  MRO:", [c.__name__ for c in Left.__mro__])
print(" ", Left().who())

### The practical use: mixins

95% of legitimate multiple inheritance is **mixins** — small classes that add
one behaviour, hold no state, and are never instantiated alone.

In [ ]:
class JsonMixin:
    """Adds to_json() to any class."""
    def to_json(self):
        return json.dumps(vars(self), default=str)


class ComparableMixin:
    """Adds comparison, provided the class defines _sort_key()."""
    def _sort_key(self):
        raise NotImplementedError("subclass must define _sort_key()")
    def __lt__(self, other): return self._sort_key() < other._sort_key()
    def __eq__(self, other): return self._sort_key() == other._sort_key()
    def __hash__(self):      return hash(self._sort_key())


class Product(JsonMixin, ComparableMixin):          # mixins LEFT, base RIGHT
    def __init__(self, code, name, price):
        self.code, self.name, self.price = code, name, price

    def _sort_key(self):
        return (self.price, self.code)

    def __repr__(self):
        return f"Product({self.code!r}, {self.price})"


items = [Product("P3", "Chair", 450.0),
         Product("P1", "Desk", 1200.0),
         Product("P2", "Lamp", 89.0)]

print("sorted  :", sorted(items))
print("to_json :", items[0].to_json())
print()
print("MIXIN ORDER MATTERS: mixins go LEFT, the concrete base goes RIGHT.")
print("MRO is left-to-right, so anything on the left can override what's")
print("on the right. Put the base first and your mixins become unreachable.")

> Deeper MRO mechanics — C3 linearisation, cooperative `__init__` chains,
> inconsistent hierarchies — are in the advanced notebook §4.3–4.5.

## 3.11 Composition over inheritance

**The most important design lesson in this section.**

In [ ]:
# ---- INHERITANCE: rigid --------------------------------------------------
class NotifierBase:
    def send(self, msg): raise NotImplementedError

class EmailNotifier(NotifierBase):
    def send(self, msg): return f"email: {msg}"

class SlackNotifier(NotifierBase):
    def send(self, msg): return f"slack: {msg}"

# Now the business wants email AND Slack together. You need a THIRD class.
# Then email + SMS. Then all three. The subclass count explodes combinatorially.


# ---- COMPOSITION: flexible ----------------------------------------------
class Email:
    def deliver(self, msg): return f"email: {msg}"

class Slack:
    def deliver(self, msg): return f"slack: {msg}"

class SMS:
    def deliver(self, msg): return f"sms:   {msg}"

class Notifier:
    """HAS channels, rather than IS a channel."""
    def __init__(self, *channels):
        self.channels = list(channels)

    def send(self, msg):
        return [c.deliver(msg) for c in self.channels]


n = Notifier(Email(), Slack(), SMS())
for line in n.send("Deploy finished"):
    print(" ", line)

# Swap behaviour AT RUNTIME — impossible with inheritance
n.channels = [Slack()]
print("quiet mode:", n.send("minor warning"))

### The decision framework

| Use **inheritance** when | Use **composition** when |
|---|---|
| A genuine "**is-a**" relationship | "**has-a**" or "uses-a" |
| The subclass can substitute for the base everywhere | behaviour varies independently |
| A framework requires it (`Exception`, `Enum`) | you want runtime swapping |
| Adding cross-cutting behaviour via mixins | combinations would explode the class count |

**The Liskov test:** if code written against the base class breaks when given
the subclass, inheritance is wrong.

> The classic failure: `class Square(Rectangle)`. A square *is* a rectangle in
> geometry — but code that does `rect.width = 5; rect.height = 3` and expects
> area 15 breaks on a square. Not an "is-a" for *this* interface.

## 3.12 Dunder methods — making your class feel built-in

In [ ]:
class Money:
    """A value object with full operator support."""
    __slots__ = ("amount", "currency")        # see §3.13

    def __init__(self, amount, currency="INR"):
        self.amount = round(float(amount), 2)
        self.currency = currency

    # ---- REPRESENTATION ----
    def __repr__(self):                       # for DEVELOPERS: unambiguous
        return f"Money({self.amount!r}, {self.currency!r})"

    def __str__(self):                        # for USERS: readable
        return f"{self.currency} {self.amount:,.2f}"

    def __format__(self, spec):               # powers f"{obj:spec}"
        return f"{self.currency}{self.amount:.0f}" if spec == "short" else format(str(self), spec)

    # ---- EQUALITY AND HASHING: define TOGETHER ----
    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented             # let Python try the other side
        return (self.amount, self.currency) == (other.amount, other.currency)

    def __hash__(self):
        return hash((self.amount, self.currency))   # hash the SAME fields you compare

    # ---- ORDERING ----
    def __lt__(self, other):
        if not isinstance(other, Money) or self.currency != other.currency:
            return NotImplemented
        return self.amount < other.amount

    # ---- ARITHMETIC ----
    def __add__(self, other):
        if isinstance(other, Money) and self.currency == other.currency:
            return Money(self.amount + other.amount, self.currency)
        return NotImplemented

    def __mul__(self, k):
        return Money(self.amount * k, self.currency) if isinstance(k, (int, float)) else NotImplemented

    def __rmul__(self, k):                    # enables  3 * money
        return self.__mul__(k)

    def __neg__(self):  return Money(-self.amount, self.currency)
    def __abs__(self):  return Money(abs(self.amount), self.currency)
    def __bool__(self): return self.amount != 0


a, b = Money(1200.50), Money(349.50)
print("repr        :", repr(a))
print("str         :", a)
print("format      :", f"{a:short}  |  {a:>18}")
print("a + b       :", a + b)
print("a * 3       :", a * 3)
print("3 * a       :", 3 * a, "  (via __rmul__)")
print("-a, abs(-a) :", -a, abs(-a))
print("a > b       :", a > b, "  (from __lt__)")
print("a == Money(1200.50):", a == Money(1200.50))
print("bool(Money(0)):", bool(Money(0)))
print("in a set    :", {a, Money(1200.50)})        # one element — equal AND same hash
print("sorted      :", sorted([a, b, Money(99)]))

### The `__eq__` / `__hash__` contract

> **If `a == b` then `hash(a)` must equal `hash(b)`.** Break it and dicts and
> sets misbehave silently.
>
> **Defining `__eq__` sets `__hash__ = None`**, making instances unhashable.
> That's Python protecting you — it forces you to define both together.

### Return `NotImplemented`, not `False`

`NotImplemented` tells Python "I can't handle this — try the other operand's
method." Returning `False` instead breaks symmetric comparison with subclasses
and unrelated types.

### The protocol map

| Protocol | Dunders |
|---|---|
| Representation | `__repr__` `__str__` `__format__` |
| Equality | `__eq__` `__hash__` |
| Ordering | `__lt__` `__le__` `__gt__` `__ge__` |
| Truthiness | `__bool__` (falls back to `__len__`) |
| Container | `__len__` `__getitem__` `__setitem__` `__contains__` |
| Iteration | `__iter__` `__next__` |
| Arithmetic | `__add__` `__sub__` `__mul__` + `__r*__` + `__i*__` |
| Callable | `__call__` |
| Context manager | `__enter__` `__exit__` |

## 3.13 `__slots__` — memory

In [ ]:
class WithDict:
    def __init__(self, a, b): self.a, self.b = a, b

class WithSlots:
    __slots__ = ("a", "b")                    # no __dict__ is created
    def __init__(self, a, b): self.a, self.b = a, b

def deep_size(o):
    return sys.getsizeof(o) + (sys.getsizeof(o.__dict__) if hasattr(o, "__dict__") else 0)

d, s = WithDict(1, 2), WithSlots(1, 2)
print(f"WithDict  : {deep_size(d):>4} bytes/object")
print(f"WithSlots : {deep_size(s):>4} bytes/object")
print(f"over 1,000,000 objects: ~{(deep_size(d)-deep_size(s))/1e6:.1f} MB saved")
print()
# Slots also catch typos
try:
    s.c = 3
except AttributeError as e:
    print("typo protection:", e)
d.c = 3
print("without slots, the typo silently creates:", d.c)

**Use `__slots__` for:** value objects you create in large numbers.
**Skip it for:** config objects, singletons, anything needing dynamic attributes.

**Caveats:** subclasses need their own `__slots__` or they regain a `__dict__`;
it breaks `cached_property`; you can't have a class-level default with the same
name.

## 3.14 Dataclasses — the modern shortcut

Most classes are just "hold these fields and compare them". `@dataclass`
generates the boilerplate.

In [ ]:
from dataclasses import dataclass, field, replace, asdict

# ---- BY HAND: 25 lines ----
class ProductManual:
    def __init__(self, code, name, price, tags=None):
        self.code, self.name, self.price = code, name, price
        self.tags = tags if tags is not None else []
    def __repr__(self):
        return (f"ProductManual(code={self.code!r}, name={self.name!r}, "
                f"price={self.price!r}, tags={self.tags!r})")
    def __eq__(self, other):
        if not isinstance(other, ProductManual): return NotImplemented
        return ((self.code, self.name, self.price, self.tags) ==
                (other.code, other.name, other.price, other.tags))


# ---- AS A DATACLASS: 6 lines, and better ----
@dataclass(slots=True, frozen=True, kw_only=True, order=True)
class Product:
    code: str
    name: str
    price: float = 0.0
    tags: tuple = ()
    notes: str = field(default="", compare=False, repr=False)   # excluded from == and repr

    def __post_init__(self):
        if self.price < 0:
            raise ValueError("price cannot be negative")


p = Product(code="P1", name="Desk", price=1200.0, tags=("office",))
print(p)                                       # __repr__ generated
print("equality :", p == Product(code="P1", name="Desk", price=1200.0, tags=("office",)))
print("hashable :", hash(p) is not None)       # frozen=True gives __hash__
print("sortable :", sorted([p, Product(code="P2", name="Lamp", price=89.0)])[0].name)

print("\nreplace  :", replace(p, price=999.0)) # the frozen-safe way to 'modify'
print("asdict   :", asdict(p))

try:
    p.price = 50
except Exception as e:
    print("frozen   :", type(e).__name__, e)

try:
    Product(code="P3", name="Bad", price=-10)
except ValueError as e:
    print("validated:", e)

| Flag | Gives you |
|---|---|
| `slots=True` | no `__dict__` — less memory, faster access. **Default choice.** |
| `frozen=True` | immutable + automatic `__hash__` |
| `kw_only=True` | callers must name arguments — prevents transposed values |
| `order=True` | `__lt__` etc. from field order |
| `field(compare=False)` | exclude from `==` |
| `field(repr=False)` | hide from logs — **use for secrets** |
| `field(default_factory=list)` | **the correct way to default a mutable** |

> **Use `@dataclass` for anything that's mostly data.** Write a full class only
> when behaviour dominates. And reach for **Pydantic** at system boundaries
> where you need runtime validation of external input.

## 3.15 SOLID, with examples

Five principles. Here's each one broken, then fixed.

In [ ]:
print("=" * 70)
print("S — SINGLE RESPONSIBILITY: a class should have one reason to change")
print("=" * 70)

# BAD: this class does three unrelated jobs
class ReportBad:
    def __init__(self, rows): self.rows = rows
    def total(self):    return sum(self.rows)              # business logic
    def to_html(self):  return f"<p>{self.total()}</p>"    # presentation
    def save(self, p):  Path(p).write_text(self.to_html()) # persistence
# Change the HTML style -> edit this class.
# Change to a database -> edit this class.
# Change the maths -> edit this class. THREE reasons to change.

# GOOD: three classes, one reason each
class Report:
    def __init__(self, rows): self.rows = rows
    def total(self): return sum(self.rows)

class HtmlFormatter:
    def format(self, report): return f"<p>{report.total()}</p>"

class FileWriter:
    def write(self, content, path): Path(path).write_text(content)

r = Report([10, 20, 30])
FileWriter().write(HtmlFormatter().format(r), "report.html")
print("  wrote:", Path("report.html").read_text())

In [ ]:
print("=" * 70)
print("O — OPEN/CLOSED: open for extension, closed for modification")
print("=" * 70)

# BAD: adding a shape means EDITING this function
def area_bad(shape):
    if shape["type"] == "circle":    return 3.14159 * shape["r"] ** 2
    elif shape["type"] == "square":  return shape["side"] ** 2
    # want a triangle? edit here. And every other function that switches on type.

# GOOD: adding a shape means ADDING a class. Nothing existing changes.
class Shape(ABC):
    @abstractmethod
    def area(self): ...

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return 3.14159 * self.r ** 2

class Square(Shape):
    def __init__(self, side): self.side = side
    def area(self): return self.side ** 2

class Triangle(Shape):                          # NEW — nothing else edited
    def __init__(self, b, h): self.b, self.h = b, h
    def area(self): return 0.5 * self.b * self.h

print("  total area:", round(sum(s.area() for s in [Circle(2), Square(3), Triangle(4, 5)]), 2))

In [ ]:
print("=" * 70)
print("L — LISKOV SUBSTITUTION: a subclass must work wherever the base does")
print("=" * 70)

class Rectangle:
    def __init__(self, w, h): self._w, self._h = w, h
    @property
    def width(self): return self._w
    @width.setter
    def width(self, v): self._w = v
    @property
    def height(self): return self._h
    @height.setter
    def height(self, v): self._h = v
    def area(self): return self._w * self._h

class Square(Rectangle):                        # "a square IS a rectangle"... is it?
    @Rectangle.width.setter
    def width(self, v):  self._w = self._h = v
    @Rectangle.height.setter
    def height(self, v): self._w = self._h = v

def stretch(rect):
    """Written against Rectangle. Expects w*h = 5*3 = 15."""
    rect.width = 5
    rect.height = 3
    return rect.area()

print("  stretch(Rectangle):", stretch(Rectangle(1, 1)), " expected 15")
print("  stretch(Square)   :", stretch(Square(1, 1)), "  <<< VIOLATION")
print()
print("  Square breaks code written for Rectangle. For THIS interface it is")
print("  not an 'is-a'. The fix: don't inherit — make them siblings under Shape.")

In [ ]:
print("=" * 70)
print("I — INTERFACE SEGREGATION: don't force clients to implement what they don't use")
print("=" * 70)

# BAD: one fat interface
class WorkerBad(ABC):
    @abstractmethod
    def work(self): ...
    @abstractmethod
    def eat(self): ...
    @abstractmethod
    def sleep(self): ...

class RobotBad(WorkerBad):
    def work(self):  return "working"
    def eat(self):   raise NotImplementedError("robots don't eat")   # <<< forced
    def sleep(self): raise NotImplementedError("robots don't sleep")

# GOOD: small, focused interfaces, combined as needed
class Workable(ABC):
    @abstractmethod
    def work(self): ...

class Biological(ABC):
    @abstractmethod
    def eat(self): ...

class Human(Workable, Biological):
    def work(self): return "working"
    def eat(self):  return "eating"

class Robot(Workable):                   # implements ONLY what applies
    def work(self): return "working"

print("  Human:", Human().work(), "/", Human().eat())
print("  Robot:", Robot().work(), "  (no eat() to stub out)")

In [ ]:
print("=" * 70)
print("D — DEPENDENCY INVERSION: depend on abstractions, not concretions")
print("=" * 70)

# BAD: hard-wired to one implementation
class ServiceBad:
    def __init__(self):
        self.db = "MySQLDatabase()"       # <<< cannot be swapped or mocked
    def get(self, i): return f"from {self.db}"

# GOOD: the dependency is INJECTED
class Repository(ABC):
    @abstractmethod
    def get(self, id_): ...

class MySQLRepo(Repository):
    def get(self, id_): return f"mysql row {id_}"

class InMemoryRepo(Repository):          # a test double, for free
    def __init__(self): self.data = {1: "test row"}
    def get(self, id_): return self.data.get(id_, "not found")

class Service:
    def __init__(self, repo: Repository):     # <<< injected
        self.repo = repo
    def fetch(self, id_): return self.repo.get(id_)

print("  production:", Service(MySQLRepo()).fetch(1))
print("  under test:", Service(InMemoryRepo()).fetch(1))
print()
print("  THE PAYOFF: the test needs no database, no network, no fixtures.")
print("  Dependency injection is what makes code testable.")

## 3.16 Anti-patterns to avoid

| Anti-pattern | Why it's wrong | Instead |
|---|---|---|
| **Getters/setters everywhere** | not Pythonic | plain attributes; add `@property` only when needed |
| **God class** | 1,000 lines, does everything | split by responsibility |
| **Deep inheritance** (4+ levels) | fragile, hard to follow | composition |
| **`class Utils:` of staticmethods** | that's a module with extra steps | a plain module |
| **Mutable class attribute** | shared by every instance | `__init__`, or `default_factory` |
| **Inheriting for code reuse alone** | couples you to the parent's internals | composition |
| **No `__repr__`** | unreadable logs and tracebacks | always define one |
| **`__eq__` without `__hash__`** | silently unhashable | define both |
| **Hard-coded dependencies** | untestable | inject them |

## 3.17 Section 3 recap

**The progression:**

1. A class bundles data with the operations on it
2. `__init__` sets up, `self` is the instance
3. **Mutable class attributes are shared** — put them in `__init__`
4. `_name` signals internal; `__name` mangles; nothing is truly private
5. `@property` adds validation without changing the caller's syntax
6. `@classmethod` for alternative constructors — use `cls`, not the class name
7. `super().__init__()` is **not optional**
8. Duck typing over declared interfaces; ABCs when you must enforce
9. `super()` follows the **MRO**, not "the parent"
10. **Composition over inheritance** — the single most useful rule
11. Dunder methods make your class feel built-in
12. `@dataclass(slots=True, frozen=True)` for value objects
13. SOLID, with dependency injection as the practical payoff

---

# 4. Multithreading

## 4.1 Concurrency vs parallelism

Two words people use interchangeably. They are not the same.

| | Means | Analogy |
|---|---|---|
| **Concurrency** | several tasks **in progress**, interleaved | one chef, several dishes, switching between them |
| **Parallelism** | several tasks **executing at the same instant** | several chefs, each on one dish |

**Concurrency is about structure. Parallelism is about hardware.** You can have
concurrency on a single core; parallelism needs several.

## 4.2 Process vs thread

| | Process | Thread |
|---|---|---|
| Memory | **its own, isolated** | **shared** with other threads |
| Created by | `multiprocessing` | `threading` |
| Startup cost | 10–50 ms | ~50 µs |
| Communication | pickle / pipes / shared memory | direct — same variables |
| One crashes | others survive | the whole process dies |
| Parallel CPU in Python | **yes** | **no** (on a standard build) |

**Threads share memory.** That's their advantage — no serialisation needed — and
their danger: two threads can modify the same variable simultaneously.

In [ ]:
import threading, queue, time, random
from concurrent.futures import ThreadPoolExecutor, as_completed

print("CPU cores      :", os.cpu_count())
print("active threads :", threading.active_count())
print("current thread :", threading.current_thread().name)

## 4.3 Your first thread

In [ ]:
def worker(name, seconds):
    print(f"  [{name}] starting on thread {threading.current_thread().name}")
    time.sleep(seconds)                      # stands in for I/O
    print(f"  [{name}] finished after {seconds}s")


# SEQUENTIAL
t0 = time.perf_counter()
worker("A", 0.5)
worker("B", 0.5)
print(f"sequential: {time.perf_counter() - t0:.2f}s\n")

# THREADED
t0 = time.perf_counter()
t1 = threading.Thread(target=worker, args=("C", 0.5))
t2 = threading.Thread(target=worker, args=("D", 0.5))

t1.start()                                   # start() BEGINS execution
t2.start()
t1.join()                                    # join() WAITS for it to finish
t2.join()
print(f"threaded  : {time.perf_counter() - t0:.2f}s")

### `start()` vs `run()` vs `join()`

| Call | Does |
|---|---|
| `t.start()` | creates an OS thread and runs `target` **in it** |
| `t.run()` | runs `target` in the **current** thread — no concurrency. A common mistake. |
| `t.join()` | blocks the caller until `t` finishes |
| `t.join(timeout=5)` | waits at most 5 seconds, then returns anyway |

> **Forget `join()` and your main thread may exit before the workers finish.**
> Whether that kills them depends on `daemon`.

In [ ]:
# DAEMON threads are killed when the main program exits.
# NON-DAEMON threads keep the program alive until they finish.
def slow_task(label):
    time.sleep(0.3)
    print(f"  [{label}] completed")

d = threading.Thread(target=slow_task, args=("daemon",), daemon=True)
n = threading.Thread(target=slow_task, args=("non-daemon",))
d.start(); n.start()

print("main thread does not join() either of them")
n.join()        # only waiting for the non-daemon one
print()
print("USE daemon=True for: background monitors, heartbeats, log flushers —")
print("anything that should not keep the program alive.")
print("USE daemon=False (default) for: work that MUST complete.")

## 4.4 The GIL — why threads don't speed up CPU work

The **Global Interpreter Lock** is a single mutex in CPython. **Only one thread
executes Python bytecode at a time.**

**But it is RELEASED during blocking I/O.** That distinction is everything.

| Operation | GIL released? |
|---|---|
| socket / file read or write | **yes** |
| `time.sleep` | **yes** |
| `subprocess` wait | **yes** |
| NumPy / pandas vectorised ops | **usually yes** |
| `hashlib`, `zlib` on large buffers | **yes** |
| pure-Python loops and arithmetic | **no** |

In [ ]:
def cpu_task(n=5_000_000):
    """Pure Python arithmetic — HOLDS the GIL the whole time."""
    total = 0
    for i in range(n):
        total += i * i
    return total

N = 4

t0 = time.perf_counter()
for _ in range(N):
    cpu_task()
serial = time.perf_counter() - t0

t0 = time.perf_counter()
threads = [threading.Thread(target=cpu_task) for _ in range(N)]
for t in threads: t.start()
for t in threads: t.join()
threaded = time.perf_counter() - t0

print(f"CPU-bound, sequential : {serial:.2f}s")
print(f"CPU-bound, {N} threads  : {threaded:.2f}s")
print(f"speedup               : {serial/threaded:.2f}x")
print()
print("NO SPEEDUP — often slightly WORSE, because you now pay for thread")
print("switching on top of the same total work. The GIL serialises it all.")

In [ ]:
def io_task(seconds=0.4):
    """time.sleep RELEASES the GIL — stands in for a network or DB call."""
    time.sleep(seconds)

N = 8

t0 = time.perf_counter()
for _ in range(N):
    io_task()
serial = time.perf_counter() - t0

t0 = time.perf_counter()
threads = [threading.Thread(target=io_task) for _ in range(N)]
for t in threads: t.start()
for t in threads: t.join()
threaded = time.perf_counter() - t0

print(f"I/O-bound, sequential : {serial:.2f}s")
print(f"I/O-bound, {N} threads  : {threaded:.2f}s")
print(f"speedup               : {serial/threaded:.1f}x")
print()
print("NEAR-LINEAR SPEEDUP. While one thread waits for I/O it releases the")
print("GIL, so another can run. Eight threads wait concurrently.")
print()
print("THE RULE: threads help with WAITING, not with COMPUTING.")

## 4.5 Race conditions — the GIL does not make you safe

**The most important misconception to correct.** The GIL serialises *bytecode*,
not your logical operations.

### Predict first

Four threads each move money from account A to B, 20,000 times, and each one
**checks the balance first**:

```python
if acct["a"] >= 10:
    acct["a"] -= 10
    acct["b"] += 10
```

A starts with 1,000. Can A ever go negative?

In [ ]:
acct = {"a": 1000, "b": 1000}

def transfer_unsafe():
    for _ in range(20_000):
        if acct["a"] >= 10:          # CHECK  — is there enough money?
            time.sleep(0)            # yield to the scheduler (see the note below)
            acct["a"] -= 10          # ACT    — take it
            acct["b"] += 10

threads = [threading.Thread(target=transfer_unsafe) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()

print(f"account A : {acct['a']:>8,}")
print(f"account B : {acct['b']:>8,}")
print(f"total     : {acct['a'] + acct['b']:>8,}   (started at 2,000)")
print()
print(f"A went NEGATIVE: {acct['a'] < 0}")
print()
print("THE BANK JUST ALLOWED AN OVERDRAFT, despite every thread checking the")
print("balance first. The total is still 2,000, so nothing looks obviously")
print("broken — only the negative balance reveals the bug.")

### Why it happens: CHECK-THEN-ACT

```
thread A:  if acct["a"] >= 10      -> True  (balance is 10)
thread B:  if acct["a"] >= 10      -> True  (balance is STILL 10 — A hasn't acted yet)
thread A:  acct["a"] -= 10         -> balance is now 0
thread B:  acct["a"] -= 10         -> balance is now -10
```

Both threads checked a condition that was true **at the moment they checked**,
and both acted on it. Between the check and the act, the world changed.

**Any read-modify-write or check-then-act sequence is a race.**

> ### An honest note about `time.sleep(0)`
>
> The `time.sleep(0)` forces a scheduler yield at the worst moment, which makes
> the race reproduce **every time** rather than occasionally.
>
> Without it, on CPython 3.12 this particular loop usually completes without
> interleaving, because the interpreter only checks for thread switches at
> certain bytecode boundaries and the switch interval is 5 ms. **That does not
> make it safe** — it means the bug appears under load, in production, and not
> in your test. Which is worse.
>
> **It is also version- and build-dependent.** On a free-threaded build (3.13+)
> there is no GIL serialising anything, so races that CPython 3.12 happens to
> hide become routine. Never rely on accidental atomicity.

In [ ]:
# Even the SIMPLEST increment is several bytecodes.
import dis

shared = 0
def inc():
    global shared
    shared += 1

dis.dis(inc)
print()
print("LOAD_GLOBAL, then BINARY_OP(+=), then STORE_GLOBAL. Three steps, and")
print("the GIL can switch threads between any two of them.")
print()
print("  thread A: LOAD shared   -> reads 100")
print("  thread B: LOAD shared   -> reads 100   <<< the SAME value")
print("  thread A: ADD, STORE    -> writes 101")
print("  thread B: ADD, STORE    -> writes 101   <<< A's update is LOST")
print()
print("Whether that interleaving actually HAPPENS depends on the interpreter")
print("version, the switch interval and the machine. Whether it CAN happen")
print("depends only on the bytecode — and it can.")

## 4.6 `Lock` — mutual exclusion

In [ ]:
# The SAME transfer, with a lock around the check AND the act together.
acct = {"a": 1000, "b": 1000}
lock = threading.Lock()

def transfer_safe():
    for _ in range(20_000):
        with lock:                   # ALWAYS use `with`, never manual acquire/release
            if acct["a"] >= 10:      # the CHECK and the ACT are now
                time.sleep(0)        # ATOMIC together — no other thread can
                acct["a"] -= 10      # observe the intermediate state
                acct["b"] += 10

threads = [threading.Thread(target=transfer_safe) for _ in range(4)]
t0 = time.perf_counter()
for t in threads: t.start()
for t in threads: t.join()
elapsed = time.perf_counter() - t0

print(f"account A : {acct['a']:>8,}")
print(f"account B : {acct['b']:>8,}")
print(f"total     : {acct['a'] + acct['b']:>8,}")
print(f"A negative: {acct['a'] < 0}")
print(f"time      : {elapsed:.2f}s")
print()
print("A stopped at 0 and never went below. The lock makes the whole")
print("check-then-act block ATOMIC: one thread inside at a time, and the")
print("others BLOCK at `with lock:` until it is released.")
print()
print("NOTE WHAT IS INSIDE THE LOCK: the CHECK as well as the ACT. Locking")
print("only the subtraction would not help — the check would still be stale.")

In [ ]:
# LOCK GRANULARITY matters enormously for performance
ITER = 200_000

def fine_grained(store, lk):
    for _ in range(ITER):
        with lk:                         # acquire/release 200,000 times
            store[0] += 1

def coarse_grained(store, lk):
    local = 0
    for _ in range(ITER):
        local += 1                       # unsynchronised LOCAL work
    with lk:                             # ONE critical section
        store[0] += local

for name, fn in [("lock per iteration", fine_grained),
                 ("lock once at the end", coarse_grained)]:
    store, lk = [0], threading.Lock()
    ts = [threading.Thread(target=fn, args=(store, lk)) for _ in range(4)]
    t0 = time.perf_counter()
    for t in ts: t.start()
    for t in ts: t.join()
    print(f"  {name:<22} {time.perf_counter()-t0:>6.2f}s   result {store[0]:,}")

print()
print("Same answer, very different cost. Hold the lock for as LITTLE time and")
print("as FEW times as correctness allows.")

### `Lock` vs `RLock`

In [ ]:
# A plain Lock is NOT reentrant — the same thread cannot acquire it twice.
plain = threading.Lock()
plain.acquire()
got = plain.acquire(timeout=0.3)         # timeout avoids hanging the notebook
print("plain Lock, second acquire by the SAME thread:", got, " <<< would DEADLOCK")
plain.release()

# RLock IS reentrant — useful when a locked method calls another locked method
rlock = threading.RLock()
with rlock:
    with rlock:                          # fine
        with rlock:                      # still fine
            print("RLock: re-acquired 3 times by the same thread")

class Inventory:
    """A realistic case for RLock: one public method calls another."""
    def __init__(self):
        self._lock = threading.RLock()
        self._items = {}

    def add(self, sku, qty):
        with self._lock:
            self._items[sku] = self._items.get(sku, 0) + qty

    def add_many(self, pairs):
        with self._lock:                 # acquires...
            for sku, qty in pairs:
                self.add(sku, qty)       # ...and add() acquires AGAIN. RLock is required.

inv = Inventory()
inv.add_many([("A", 5), ("B", 3), ("A", 2)])
print("inventory:", inv._items)

## 4.7 The other synchronisation primitives

| Primitive | Use for |
|---|---|
| `Lock` | mutual exclusion. **Not** reentrant |
| `RLock` | same thread may re-acquire — nested locked methods |
| `Semaphore(n)` | cap concurrency at n — connection pools, rate limits |
| `BoundedSemaphore(n)` | as above, but raises on over-release (catches bugs) |
| `Event` | one-way signal / **cooperative cancellation** |
| `Condition` | wait until a predicate becomes true |
| `Barrier(n)` | all n threads meet before any proceeds |
| `local()` | per-thread storage — **avoids sharing entirely** |

In [ ]:
# SEMAPHORE: limit concurrency without limiting thread count
sem = threading.BoundedSemaphore(3)
active = peak = 0
lk = threading.Lock()

def guarded(i):
    global active, peak
    with sem:                                 # at most 3 threads inside
        with lk:
            active += 1; peak = max(peak, active)
        time.sleep(random.uniform(0.02, 0.06))
        with lk:
            active -= 1

ts = [threading.Thread(target=guarded, args=(i,)) for i in range(12)]
for t in ts: t.start()
for t in ts: t.join()
print(f"12 threads ran; peak concurrent inside the semaphore: {peak} (limit 3)")

In [ ]:
# EVENT: the ONLY correct way to stop a thread. There is no thread.kill().
stop = threading.Event()
results = []

def pollable_worker(name):
    n = 0
    while not stop.is_set():
        n += 1
        if stop.wait(timeout=0.02):           # doubles as an INTERRUPTIBLE sleep
            break
    results.append((name, n))

ts = [threading.Thread(target=pollable_worker, args=(f"w{i}",), daemon=True)
      for i in range(3)]
for t in ts: t.start()
time.sleep(0.15)
stop.set()                                    # signal: please stop
for t in ts: t.join(timeout=1)

print("cleanly stopped:", results)
print()
print("PYTHON CANNOT FORCIBLY KILL A THREAD. No kill(), no terminate().")
print("You MUST design cancellation in: a flag the worker checks, and")
print("stop.wait(timeout=...) instead of time.sleep() so it wakes immediately.")

In [ ]:
# threading.local(): each thread gets its OWN copy. No lock needed at all.
ctx = threading.local()
seen = []

def handler(request_id):
    ctx.request_id = request_id               # invisible to other threads
    ctx.user = f"user-{request_id}"
    inner()                                   # nested call sees the same context

def inner():
    seen.append((threading.current_thread().name, ctx.request_id, ctx.user))

ts = [threading.Thread(target=handler, args=(i,), name=f"t{i}") for i in range(4)]
for t in ts: t.start()
for t in ts: t.join()

for row in sorted(seen):
    print(" ", row)
print()
print("No cross-contamination, and no locking. THE CHEAPEST SYNCHRONISATION")
print("IS NOT SHARING. This is how web frameworks store per-request context.")

## 4.8 `queue.Queue` — the preferred way to share data

**Prefer a queue to a shared list plus a lock.** `Queue` is thread-safe, handles
blocking, and gives you backpressure for free.

In [ ]:
q = queue.Queue(maxsize=5)                    # maxsize gives BACKPRESSURE
SENTINEL = object()
processed, errors = [], []

def producer(n_items):
    for i in range(n_items):
        q.put({"id": i, "payload": f"row-{i}"})   # BLOCKS if the queue is full
        time.sleep(0.004)
    for _ in range(3):
        q.put(SENTINEL)                       # one sentinel PER consumer

def consumer(name):
    while True:
        item = q.get()
        try:
            if item is SENTINEL:
                return
            if item["id"] == 7:
                raise ValueError("bad row 7")
            time.sleep(random.uniform(0.004, 0.015))
            processed.append((name, item["id"]))
        except Exception as e:
            errors.append((name, str(e)))
        finally:
            q.task_done()                     # required if you use q.join()

prod = threading.Thread(target=producer, args=(15,))
cons = [threading.Thread(target=consumer, args=(f"c{i}",)) for i in range(3)]
prod.start(); [c.start() for c in cons]
prod.join();  [c.join() for c in cons]

print(f"processed {len(processed)} items across "
      f"{len({p[0] for p in processed})} consumers")
print("errors:", errors)

> ### Two details that matter
>
> **The sentinel pattern:** put one sentinel per consumer, so each exits exactly
> once. The alternative is a shared `Event` plus `get(timeout=...)`, needed when
> consumers must react to shutdown while idle.
>
> **`maxsize` is not optional in production.** An unbounded queue turns a slow
> consumer into an out-of-memory crash. Bounding it applies backpressure to the
> producer instead.

### Queue variants

| Class | Order |
|---|---|
| `queue.Queue` | FIFO — first in, first out |
| `queue.LifoQueue` | LIFO — a stack |
| `queue.PriorityQueue` | lowest priority value first |

## 4.9 `ThreadPoolExecutor` — what you should actually use

Raw `Thread` is educational. **In real code, use the executor.** You get result
handling, exception propagation and clean shutdown.

In [ ]:
def fetch(url, fail=False):
    time.sleep(random.uniform(0.05, 0.2))
    if fail:
        raise ConnectionError(f"failed: {url}")
    return f"200 {url}"

urls = [f"/api/{i}" for i in range(6)]

# submit + as_completed: results arrive in COMPLETION order
with ThreadPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(fetch, u, fail=(i == 2)): u for i, u in enumerate(urls)}
    for fut in as_completed(futures):
        url = futures[fut]
        try:
            print("  ", fut.result())         # re-raises the worker's exception HERE
        except ConnectionError as e:
            print("   ERROR", url, "->", e)

print()
print("The `with` block calls shutdown(wait=True) on exit, so every thread is")
print("joined automatically. No manual start()/join() bookkeeping.")

In [ ]:
# map(): results in SUBMISSION order — but exceptions have a nasty property
def work(x):
    time.sleep(0.03)
    if x == 3:
        raise ValueError("bad input 3")
    return x * 2

with ThreadPoolExecutor(max_workers=4) as ex:
    it = ex.map(work, range(6))
    i = 0
    while True:
        try:
            print(f"  item {i}:", next(it))
        except StopIteration:
            print("  iterator exhausted"); break
        except ValueError as e:
            print(f"  item {i}: ValueError: {e}")
            print("  <<< items 4 and 5 are now UNREACHABLE via this iterator")
            break
        i += 1

print()
print("An exception from map() TERMINATES the iterator. Use submit() +")
print("as_completed() when you need per-item error isolation.")

In [ ]:
# You cannot forcibly cancel a RUNNING task
def slow(): time.sleep(1.5); return "done"

with ThreadPoolExecutor(max_workers=1) as ex:
    fut = ex.submit(slow)
    try:
        fut.result(timeout=0.2)
    except TimeoutError:
        print("timed out waiting for the result")
    print("cancel() succeeded:", fut.cancel(), " <<< False — already running")
    print("cancel() only works on tasks still QUEUED, never on running ones.")
    fut.result()                              # wait for clean shutdown

print()
print("Design COOPERATIVE cancellation with an Event, as in 4.7.")

> **Sizing:** `ThreadPoolExecutor` defaults to `min(32, cpu_count + 4)`. For
> I/O-heavy work that's often too low — 50–200 threads is reasonable when
> they're all waiting on network. For CPU-bound work, threads are the wrong tool
> entirely.

## 4.10 Deadlock

In [ ]:
lock_a, lock_b = threading.Lock(), threading.Lock()
deadlocked = threading.Event()

def t1():
    with lock_a:
        time.sleep(0.05)
        if not lock_b.acquire(timeout=0.3):   # timeout turns a HANG into an ERROR
            deadlocked.set(); return
        lock_b.release()

def t2():
    with lock_b:
        time.sleep(0.05)
        if not lock_a.acquire(timeout=0.3):
            deadlocked.set(); return
        lock_a.release()

a, b = threading.Thread(target=t1), threading.Thread(target=t2)
a.start(); b.start(); a.join(); b.join()

print("deadlock detected via timeout:", deadlocked.is_set())
print()
print("WHY: t1 holds A and wants B. t2 holds B and wants A. Neither can")
print("proceed, and neither will ever release. Without the timeout, this")
print("would hang FOREVER with no error.")

In [ ]:
# THE FIX: acquire locks in a GLOBAL CONSISTENT ORDER.
locks = {"a": threading.Lock(), "b": threading.Lock()}

def ordered_work(names):
    for name in sorted(names):                # <<< the entire fix
        locks[name].acquire()
    try:
        time.sleep(0.02)
    finally:
        for name in sorted(names, reverse=True):
            locks[name].release()

ts = [threading.Thread(target=ordered_work, args=(["a", "b"],)) for _ in range(4)]
t0 = time.perf_counter()
for t in ts: t.start()
for t in ts: t.join()
print(f"no deadlock, completed in {time.perf_counter()-t0:.3f}s")
print()
print("Sorting the names means EVERY thread takes 'a' before 'b'. A cycle")
print("becomes impossible. Same rule as the SQL deadlock fix from Day 3:")
print("always touch resources in the same order.")

### Four deadlock defences, in order of preference

1. **Don't share** — `threading.local`, message passing via `Queue`, immutable data
2. **One lock** — coarser but correct beats fine-grained and broken
3. **Global lock ordering** — sort by a stable key before acquiring
4. **Timeouts** — `acquire(timeout=...)` turns a silent hang into a loud error

## 4.11 What's atomic, and what isn't

In [ ]:
print("OPERATIONS THAT ARE EFFECTIVELY ATOMIC (single bytecode under the GIL):")
print("  list.append(x)          dict[k] = v          set.add(x)")
print("  list.pop()              dict.pop(k)          x = y  (simple assignment)")
print()
print("OPERATIONS THAT ARE NOT:")
print("  counter += 1                 LOAD, ADD, STORE")
print("  d[k] = d.get(k, 0) + 1       read THEN write")
print("  if k not in d: d[k] = v      check THEN act  <<< classic race")
print("  lst[0] = lst[0] * 2          read THEN write")
print()
print("THE PATTERN: any READ-MODIFY-WRITE sequence is a race. Anything that")
print("CHECKS a condition and then ACTS on it is a race.")
print()
print("AND DO NOT RELY ON THIS. The atomicity above is a CPython implementation")
print("detail, not a language guarantee. On a free-threaded build (3.13+) it may")
print("not hold. Use a lock and state your intent.")

## 4.12 Choosing a concurrency model

```mermaid
flowchart TD
  A["is the work WAITING on something<br/>external? network, disk, DB"] -->|yes| B{"how many<br/>concurrent operations?"}
  A -->|"no — CPU-bound"| C{"what does the hot loop do?"}
  B -->|"hundreds or thousands"| D["asyncio"]
  B -->|"a few dozen, or blocking libraries"| E["THREADS"]
  C -->|"NumPy / pandas / C"| E
  C -->|"pure Python"| F["PROCESSES"]
```

| | Parallel CPU | Memory | Startup | Sharing | Ceiling |
|---|---|---|---|---|---|
| `threading` | **no** | shared | ~50 µs | direct, needs locks | ~100s |
| `multiprocessing` | **yes** | separate | 10–50 ms | pickle / shared_memory | ~CPU count |
| `asyncio` | no | shared | ~1 µs | direct, single-threaded | **10,000s** |

**The decision table:**

| Workload | Choose |
|---|---|
| 1,000 HTTP calls | **asyncio** |
| A handful of DB queries, blocking driver | **threads** |
| CPU-bound pure Python | **processes** |
| CPU-bound NumPy/pandas | **threads** (they release the GIL) |
| Large arrays, many workers | processes + `shared_memory` |
| Reading 50 files from disk | **threads** |

> **Free-threaded Python (3.13 experimental, 3.14 officially supported)** removes
> the GIL, so CPU-bound threads can finally scale. **But your code still needs
> locks** — removing the GIL removes accidental protection, it doesn't make
> `counter += 1` atomic. Covered in the advanced notebook §12.10.

## 4.13 Section 4 recap

**The rules**

1. **Threads help with waiting, not computing.** I/O-bound → near-linear speedup; CPU-bound → none.
2. **The GIL does not make you thread-safe.** `counter += 1` is three bytecodes and loses updates.
3. **Always use `with lock:`**, never manual `acquire()`/`release()`.
4. **Hold locks briefly and rarely** — granularity dominates performance.
5. **`RLock` when a locked method calls another locked method.**
6. **You cannot kill a thread.** Design cooperative cancellation with an `Event`.
7. **Prefer `Queue` to shared state**, and always set `maxsize`.
8. **Use `ThreadPoolExecutor`**, not raw `Thread`, in real code.
9. **Prevent deadlock with a consistent lock order**, and add timeouts.
10. **The cheapest synchronisation is not sharing** — `threading.local`, queues, immutable data.

---

# 5. Interview questions

## File I/O

1. Text mode vs binary mode — when must you use binary?
2. Why pass `encoding="utf-8"` explicitly instead of relying on the default?
3. You read an Excel-exported CSV and the first column name has junk in it. Why?
4. How do you process a 50 GB file on a 4 GB machine?
5. What is an atomic write, and why does a plain `open(path, "w")` not give you one?
6. When would you use `ExitStack`?

## Pickle

7. What can pickle serialise that JSON cannot?
8. Why can't you pickle a lambda?
9. **What's the security risk of `pickle.load`?**
10. What are `__getstate__` and `__setstate__` for?
11. Pickle vs JSON vs Parquet — when each?
12. Why does sklearn recommend `joblib` over `pickle`?

## OOP

13. What is `self`, and what does `obj.method()` actually become?
14. Instance vs class attributes — and what breaks with a mutable class attribute?
15. `_name` vs `__name` — what does Python actually do?
16. When would you add a `@property`?
17. `classmethod` vs `staticmethod` vs a module-level function.
18. Why must `super().__init__()` be called?
19. Duck typing vs an ABC — when do you need the ABC?
20. **Composition vs inheritance — what's your decision rule?**
21. What's the `__eq__`/`__hash__` contract, and what breaks if you violate it?
22. What does `__slots__` buy, and what does it cost?
23. Explain the Liskov Substitution Principle with an example that violates it.
24. What is dependency injection, and why does it matter for testing?

## Threading

25. Concurrency vs parallelism.
26. **What does the GIL protect, and name four operations that release it.**
27. Show why `counter += 1` races despite the GIL.
28. `Lock` vs `RLock` — give a case that needs RLock.
29. How do you stop a running thread? (Trick question.)
30. Why must a production `queue.Queue` have a `maxsize`?
31. Four ways to prevent deadlock, in order of preference.
32. 5,000 concurrent HTTP calls — threads, processes or asyncio? Now the work is CPU-bound on NumPy arrays — what changes?
33. Does free-threaded Python remove the need for locks?

---

# The ten things to remember

1. **Binary mode for anything you didn't hand-write**, and always pass `encoding=`.
2. **Write to `.tmp`, then `os.replace`** — a partial file that looks complete is worse than a crash.
3. **`for line in f`, never `readlines()`** on a large file.
4. **Unpickling executes code.** Never on data you didn't create.
5. **Pickle stores an import path, not code** — which is why lambdas fail and renaming a class breaks old pickles.
6. **Mutable class attributes are shared by every instance.** Put them in `__init__`.
7. **`super().__init__()` is not optional.**
8. **Composition over inheritance** — the single most useful OOP rule.
9. **The GIL serialises bytecode, not your logic.** Threads still need locks.
10. **Threads for waiting, processes for computing, asyncio for thousands of waits.**